# MeSH Co-occurrence Network: Topical Structure of the Corpus

This notebook maps the topical structure of the corpus through MeSH descriptor co-occurrence: which medical subjects appear together on the same articles, which sit at the centre of the research landscape, and how that structure differs by era.

It builds on the EDA (`03_eda.ipynb`), which established two facts this notebook depends on. First, generic descriptors dominate ("Humans", "Female", "Male", "Adult", and so on); they tag nearly every article and carry no topical signal, so they are stripped before any co-occurrence is computed. Second, MeSH depth shifts after about 2019 (mean descriptors per article falls from roughly 13 to roughly 8 due to NLM automated indexing), so co-occurrence density is not comparable across that boundary, and eras are analysed separately.

Runs on the published metadata (no abstracts needed). The loader mirrors the EDA's Option A / Option B pattern.

## Setup

In [ ]:
import os, glob, collections, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: pick ONE option (same pattern as 03_eda)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input; attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "mesh_descriptors"])
df = df[df["year"] <= 2025].copy()                       # drop live edge, as in 03_eda
print(f"loaded {len(df):,} records")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs; dedup did not run"
df.head(3)

## 1. Strip generic descriptors

The most frequent descriptors are demographic and structural, not topical. They co-occur with everything, so leaving them in would make every article look connected and drown out the real topical signal. The removal is justified by evidence, not assumption: each candidate is checked against its document frequency (the share of articles carrying it), and the set is curated by MeSH category. The two helper functions below are reused across rounds.

In [ ]:
def diagnose(column, label, threshold=40, top_n=40):
    """Document-frequency diagnostic for a descriptor-list column. Prints the table,
    the coverage-band counts, and plots the top_n. Returns the frequency Series."""
    N = len(df)
    dfq = collections.Counter()
    for lst in df[column]:
        if isinstance(lst, (list, np.ndarray)):
            dfq.update(set(lst))
    pct = (pd.Series(dfq) / N * 100).sort_values(ascending=False)

    print(f"=== {label}: {len(pct):,} distinct descriptors ===")
    print(pct.head(25).round(1).to_string())
    print("\nCoverage bands:")
    for band in [90, 75, 50, 40, 30, 20, 10, 5]:
        print(f"  >= {band:>2}%: {(pct >= band).sum():>4} descriptors")

    plt.figure(figsize=(10, 9))
    top = pct.head(top_n)[::-1]
    sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
    if threshold:
        plt.axvline(threshold, color="#e07a5f", ls="--", lw=1.5, label=f"{threshold}% mark")
        plt.legend()
    plt.title(f"Descriptor document frequency: {label}")
    plt.xlabel("% of articles carrying the descriptor"); plt.ylabel("")
    plt.tight_layout(); plt.show()
    return pct


def apply_strip(generic_set):
    """Strip generic_set from mesh_descriptors into mesh_topical, and report the damage."""
    def topical_only(lst):
        if not isinstance(lst, (list, np.ndarray)):
            return []
        return [d for d in lst if d not in generic_set]

    df["mesh_topical"] = df["mesh_descriptors"].map(topical_only)
    df["n_topical"] = df["mesh_topical"].map(len)

    n0 = (df["n_topical"] == 0).sum()
    n1 = (df["n_topical"] == 1).sum()
    n2 = (df["n_topical"] >= 2).sum()
    print(f"removed {len(generic_set)} generic terms. Resulting topical counts:")
    print(f"  0 topical (emptied by the strip): {n0:>9,}  ({n0/len(df)*100:.1f}%)")
    print(f"  1 topical (cannot co-occur):      {n1:>9,}  ({n1/len(df)*100:.1f}%)")
    print(f"  >=2 topical (usable):             {n2:>9,}  ({n2/len(df)*100:.1f}%)")
    print(f"  mean topical/article: {df['n_topical'].mean():.2f}")

### 1a. Round 1: inspect the raw frequencies
The diagnostic ranks every descriptor by document frequency. Look for which terms sit at the top and whether there is a natural break between near-ubiquitous terms and genuinely topical ones.

In [ ]:
pct_raw = diagnose("mesh_descriptors", "round 1: raw")

### 1b. Round 1 strip: demographic check-tags and study-design qualifiers
The set below is taken from the table above. Only "Humans" is universal (100%, guaranteed by the human-subject filter); after that the frequency declines smoothly, so a single threshold cannot separate generic from topical. The set is therefore curated by MeSH category: check-tags, age-band qualifiers, and study-design qualifiers. These describe the population and methodology of a study, not its subject.

In [ ]:
# Justified by the document-frequency table above AND by MeSH category:
# these are demographic check-tags and study-design qualifiers, not topical subjects.
GENERIC = {
    # Check-tags (MeSH administrative population descriptors), top of the frequency table:
    "Humans",            # 100%, guaranteed by the human-subject filter, zero information
    "Female", "Male",    # sex check-tags
    "Animals", "Mice",   # organism check-tags
    "Pregnancy",         # check-tag
    # Age-band qualifiers (demographic, not topical):
    "Adult", "Middle Aged", "Aged", "Adolescent", "Child", "Young Adult",
    "Aged, 80 and over", "Child, Preschool", "Infant",
    # Study-design / methodology qualifiers (characterize method, not subject):
    "Retrospective Studies", "Prospective Studies", "Cross-Sectional Studies",
    "Follow-Up Studies", "Time Factors", "Surveys and Questionnaires",
    "Treatment Outcome", "Risk Factors",
}

apply_strip(GENERIC)

### 1c. Round 2: re-inspect what survived
Re-running the diagnostic on the stripped column shows what is now at the top. If generic-looking terms remain, they are added in the next round. This is the manual, evidence-led loop: strip, look, strip again, until the top of the list is genuinely topical.

In [ ]:
pct_stripped = diagnose("mesh_topical", "round 2: after first strip")

### 1d. Round 2 strip: remaining epidemiological and study-design qualifiers
A second pass removes the methodological survivors revealed by round 2 (cohort/case-control designs, prevalence/incidence measures, and similar). The empties report after each strip shows the cost: how many articles fall below two topical descriptors and so cannot contribute to co-occurrence.

In [ ]:
GENERIC |= {
    # epidemiological / study-design qualifiers (method, not subject):
    "Cohort Studies", "Case-Control Studies",
    "Reproducibility of Results", "Risk Assessment", "Sensitivity and Specificity",
    "Severity of Illness Index", "Prevalence", "Incidence", "Prognosis",
    "Age Factors",                          # demographic qualifier, sibling of the age bands
    "Infant, Newborn",                      # age check-tag, sibling of "Infant"
    "Molecular Sequence Data",              # administrative data tag
    # methods-substrate / connective terms revealed as over-central by the network (4a):
    "Biomarkers", "Models, Biological", "Disease Progression",
}
apply_strip(GENERIC)                        # re-check empties
pct3 = diagnose("mesh_topical", "round 3: after second strip")

**What this shows:** by round 3 the empties cost is negligible (almost every article keeps two or more topical descriptors) and the top of the list is genuinely topical: diseases, anatomy, biological processes, and techniques rather than demographic or methodological tags. Nothing dominates (no term sits far above the rest), so no single descriptor can distort the co-occurrence network. This is a good place to stop stripping; removing more would trade defensibility for marginal tidiness.

### 1e. Sanity check: is "United States" topical or a hub?
"United States" stays near the top after stripping. It is kept only if it behaves topically. The test: what does it co-occur with? If it pairs with specific subjects (health policy, disparities, epidemiology) it is topical; if it pairs with everything evenly it is a hub and should be removed.

In [ ]:
us_pairs = collections.Counter()
for lst in df["mesh_topical"]:
    s = set(lst)
    if "United States" in s:
        us_pairs.update(s - {"United States"})
print("United States co-occurs most with:")
for term, c in us_pairs.most_common(15):
    print(f"  {term:<35} {c:,}")

**What this shows:** "United States" co-occurs with a coherent, specific cluster: Medicare, Medicaid, Socioeconomic Factors, Health Services Accessibility, Ethnicity, and related terms. That is US health-services and health-disparities research, a genuine topic, not a flat smear across every subject. It is therefore kept as a topical descriptor.

## 2. Build the co-occurrence counts

For each article, every unordered pair of its topical descriptors is counted once. `itertools.combinations` over the de-duplicated descriptor set per article is the memory-safe way to do this; it never materialises an exploded cross-join of millions of rows. A progress bar is shown because this passes over every record.

In [ ]:
from tqdm.auto import tqdm

def cooccurrence(series, show_progress=True):
    """Return (pair_counts, term_freq) over an iterable of descriptor lists."""
    pair = collections.Counter()
    freq = collections.Counter()
    it = tqdm(series, desc="counting co-occurrences", total=len(series)) if show_progress else series
    for lst in it:
        terms = sorted(set(lst))
        freq.update(terms)
        pair.update(itertools.combinations(terms, 2))   # each unordered pair once
    return pair, freq

pair_counts, term_freq = cooccurrence(df["mesh_topical"])
print(f"distinct topical descriptors: {len(term_freq):,}")
print(f"distinct co-occurring pairs:  {len(pair_counts):,}")
print("\nmost frequent topical descriptors:")
for t, c in term_freq.most_common(10):
    print(f"  {t:<35} {c:,}")
print("\nstrongest raw co-occurring pairs:")
for (a, b), c in pair_counts.most_common(8):
    print(f"  {a}  +  {b}  :  {c:,}")

**What this shows:** the strongest raw pairs are all recognisable biomedical associations (a disease and its virus, an organ and the technique that images it, a US-disparities cluster). The absence of demographic noise at the top confirms the stripping worked. Raw counts favour frequent terms, so section 3 normalizes to surface genuinely specific associations rather than merely frequent ones.

### 2a. Checks before building the matrix
Two quick checks. The maximum descriptors on one article bounds the pair explosion (a paper with k descriptors makes k-choose-2 pairs). The pair-count distribution shows how many pairs are noise (most occur only once), which justifies the edge threshold used later in the network.

In [ ]:
print("max topical descriptors on one article:", df["n_topical"].max())

counts = np.array(list(pair_counts.values()))
print(f"\ntotal pairs: {len(counts):,}")
for thresh in [1, 2, 5, 10, 25, 50, 100, 500, 1000]:
    n = (counts >= thresh).sum()
    print(f"  pairs with count >= {thresh:>4}: {n:>10,}  ({n/len(counts)*100:.1f}%)")

singletons = (counts == 1).sum()
print(f"\npairs occurring exactly once: {singletons:,} ({singletons/len(counts)*100:.0f}%)")

**What this shows:** about half of all distinct pairs occur exactly once; these are incidental and carry no structure. The counts fall off as a smooth power law with no sharp cliff, so the network edge threshold is chosen for readability (keep strong links only) rather than at a natural break. The maximum descriptors per article is modest, so no single paper dominates the pair counts.

## 3. Co-occurrence matrix and normalized association

Raw co-occurrence is dominated by frequency: a common descriptor co-occurs with everything simply because it is common. To find meaningful associations, the counts are normalized with the Jaccard index, `co(a,b) / (freq(a) + freq(b) - co(a,b))`, which measures how often two terms appear together relative to how often either appears at all. High Jaccard means a genuinely specific pairing.

In [ ]:
TOPN = 20
top_terms = [t for t, _ in term_freq.most_common(TOPN)]
idx = {t: i for i, t in enumerate(top_terms)}

raw = np.zeros((TOPN, TOPN))
jac = np.zeros((TOPN, TOPN))
for (a, b), c in pair_counts.items():
    if a in idx and b in idx:
        i, j = idx[a], idx[b]
        raw[i, j] = raw[j, i] = c
        union = term_freq[a] + term_freq[b] - c
        v = c / union if union else 0.0
        jac[i, j] = jac[j, i] = v

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(raw, xticklabels=top_terms, yticklabels=top_terms, cmap="Blues",
            ax=axes[0], cbar_kws={"label": "co-occurrence count"})
axes[0].set_title("Raw co-occurrence (top 20 topical descriptors)")
sns.heatmap(jac, xticklabels=top_terms, yticklabels=top_terms, cmap="Reds",
            ax=axes[1], cbar_kws={"label": "Jaccard association"})
axes[1].set_title("Normalized association (Jaccard)")
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
plt.tight_layout(); plt.show()

**What this shows:** the raw matrix (left) highlights frequent co-occurrences; the Jaccard matrix (right) highlights specific ones, pairs that genuinely belong together rather than pairs that are merely both common. Frequent-but-unspecific terms (such as "United States") dim on the right while tightly coupled subjects stand out. Read the right panel for topical structure and the left for volume.

## 4. The co-occurrence network

Treating descriptors as nodes and co-occurrences as weighted edges turns the matrix into a network, where centrality identifies the subjects that bridge the most research areas. Edges below a minimum count are dropped to keep the graph readable and to remove incidental pairings; the threshold is set from the pair-count distribution in section 2a. Requires `networkx`.

In [ ]:
try:
    import networkx as nx
except ImportError:
    print("networkx not installed; run `pip install networkx` to build the graph.")
    nx = None

if nx is not None:
    GRAPH_TOPN = 40
    MIN_EDGE = 500          # from the pair-count distribution: drops the ~50% singleton noise
    graph_terms = set(t for t, _ in term_freq.most_common(GRAPH_TOPN))

    G = nx.Graph()
    for t in graph_terms:
        G.add_node(t, freq=term_freq[t])
    for (a, b), c in pair_counts.items():
        if a in graph_terms and b in graph_terms and c >= MIN_EDGE:
            G.add_edge(a, b, weight=c)
    # keep only the largest connected component for a clean layout
    if G.number_of_nodes():
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()

    print(f"network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    cent = nx.degree_centrality(G)
    print("\nmost central descriptors (bridge the most subjects):")
    for t, v in sorted(cent.items(), key=lambda x: -x[1])[:8]:
        print(f"  {t:<35} {v:.3f}")

**What this shows:** the most central descriptors are the subjects that bridge the most research areas. After the round-2 strip removed the connective methods terms (Biomarkers, Models Biological, Disease Progression), centrality resolves to genuine subjects (broad disease categories, anatomy, and high-volume conditions) rather than methodological tags. Centrality here means "studied alongside many other subjects", not biological importance.

### 4b. Topical communities (clusters of co-occurring subjects)
Rather than computing three-way or higher combinations explicitly, community detection on the pair network finds groups of descriptors that all interconnect, the multi-term themes of the corpus. Each community is a set of subjects studied together (a disease and its mechanisms, or a methods family). Nodes are coloured by community.

In [ ]:
if nx is not None and G.number_of_nodes():
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G, weight="weight"))

    print(f"found {len(communities)} topical communities:\n")
    node_comm = {}
    for i, com in enumerate(communities):
        members = sorted(com, key=lambda t: -term_freq[t])
        node_comm.update({n: i for n in com})
        print(f"community {i+1} ({len(com)} terms): {', '.join(members[:8])}"
              + (" ..." if len(com) > 8 else ""))

    palette = plt.colormaps["tab10"]
    colors = [palette(node_comm[n] % 10) for n in G.nodes()]

    pos = nx.spring_layout(G, k=0.6, seed=42, weight="weight")
    sizes = [400 + 6000 * cent[n] for n in G.nodes()]
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    wmax = max(weights) if weights else 1
    widths = [0.3 + 3.5 * (w / wmax) for w in weights]

    plt.figure(figsize=(14, 11))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.2, edge_color="#888")
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, alpha=0.9)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.title("MeSH co-occurrence network, coloured by topical community")
    plt.axis("off"); plt.tight_layout(); plt.show()

**What this shows:** the network resolves into distinct topical communities; each colour is a cluster of subjects that co-occur strongly with each other. These are the corpus's research themes, recovered from pairwise co-occurrence without needing explicit three-way counts: if a group of terms all pair with each other, they fall into one community. The largest, most central node in each cluster is its anchor subject.

## 5. How the structure shifts by era

Because MeSH depth drops after about 2019 (EDA section 6c), the number of descriptors per article, and therefore raw co-occurrence density, is not comparable across that boundary. The eras below are split on that real structural boundary rather than an arbitrary year. MeSH coverage is 100% in all years, so there is no reason to discard the early data; the full range is used. Associations are read from the normalized (Jaccard) view, which is far less sensitive to depth than raw counts.

In [ ]:
# Split on the real structural boundary: the 2019/2020 MeSH-indexing change (EDA 6c).
# MeSH coverage is 100% in all years, so early data is kept, not discarded.
eras = {
    "1994-2009": (1994, 2009),    # early period
    "2010-2019": (2010, 2019),    # mature manual indexing, stable depth ~13
    "2020-2025": (2020, 2025),    # automated indexing, depth ~8
}

def top_pairs_for(sub, n=10):
    pc, fr = cooccurrence(sub["mesh_topical"], show_progress=False)
    scored = []
    for (a, b), c in pc.items():
        union = fr[a] + fr[b] - c
        if union and c >= 20:
            scored.append(((a, b), c / union, c))
    scored.sort(key=lambda x: -x[1])
    return scored[:n]

for label, (lo, hi) in eras.items():
    sub = df[(df["year"] >= lo) & (df["year"] <= hi)]
    print(f"\n=== {label}  ({len(sub):,} articles, mean topical/article {sub['n_topical'].mean():.1f}) ===")
    for (a, b), j, c in top_pairs_for(sub):
        print(f"  {j:.3f}  {a}  +  {b}   (n={c:,})")

**What this shows:** the strongest normalized pairs in each era reveal which subject couplings define that period. Because this uses Jaccard (a ratio) rather than raw counts, the post-2019 depth drop does not distort the comparison: it measures specificity of association, which is depth-robust, not volume, which is not.

### 5b. What emerged in the most recent era
The plain top-pairs lists mostly repeat the same dominant couplings across eras. More informative is what is strong now but was absent before: pairs with high Jaccard in 2020-2025 that barely registered in 2010-2019. This surfaces genuinely new associations (for example the pandemic cluster) rather than the persistent ones.

In [ ]:
def pair_jaccard(sub):
    pc, fr = cooccurrence(sub["mesh_topical"], show_progress=False)
    out = {}
    for p, c in pc.items():
        union = fr[p[0]] + fr[p[1]] - c
        if c >= 20 and union:
            out[p] = c / union
    return out

j_early = pair_jaccard(df[df["year"].between(2010, 2019)])
j_late  = pair_jaccard(df[df["year"].between(2020, 2025)])

emerged = {p: j_late[p] for p in j_late if j_late[p] > 0.05 and j_early.get(p, 0) < 0.01}
print("pairs that emerged in 2020-2025 (strong now, absent before):")
for (a, b), j in sorted(emerged.items(), key=lambda x: -x[1])[:15]:
    print(f"  {j:.3f}  {a}  +  {b}")

**What this shows:** these couplings became prominent only in the most recent era. Pandemic-related terms appearing here and not earlier are the clearest example. Because the comparison is on Jaccard rather than raw counts, the list reflects newly specific associations, not simply terms that grew with overall volume.

## 6. Summary and caveats

### What this notebook produced
This notebook mapped the topical structure of the corpus through MeSH co-occurrence. After stripping generic descriptors on documented evidence, it computed normalized (Jaccard) associations, built a co-occurrence network whose central nodes are the subjects that bridge the most research areas, decomposed that network into topical communities, and compared structure across three eras split on the 2019/2020 indexing boundary.

### What the results show
The co-occurrence counts are clean: the strongest raw pairs are all recognisable biomedical associations (a disease and its virus, an organ and the technique that images it, a US health-disparities cluster), with no demographic noise at the top, which confirms the stripping worked. The network centrality, after removing the connective methods terms the network itself flagged as over-central, resolves to genuine high-volume subjects (Neoplasms, Breast Neoplasms, Brain, Mutation) plus United States as the anchor of the health-services cluster. The clearest era finding comes from the emerged-pairs view: coronavirus and pandemic terms (Coronavirus Infections with Pneumonia Viral, COVID-19 with SARS-CoV-2, COVID-19 Serotherapy with Passive Immunization) appear strongly in 2020-2025 and are absent before, which is the pandemic showing up in the data.

### Caveats

**Jaccard surfaces tight vocabulary couplings, not broad themes.** The highest-Jaccard pairs in each era are dominated by near-synonyms and definitional links (a drug and its class, a virus and its infection, a cell line and its source organism, for example "CHO Cells" with "Cricetulus" or "Helicobacter Infections" with "Helicobacter pylori"). These score near 1.0 because the two terms almost always appear together. This is correct behaviour for the metric, but it means the per-era top-pairs lists describe MeSH's internal vocabulary structure more than they describe research trends. For genuine trends, read the emerged-pairs view (5b) and the network communities (4b), which filter out the persistent definitional pairs.

**Stripping is a judgement call, refined by the network.** The GENERIC set is the demographic and study-design terms identified by document frequency, plus the connective methods terms (Biomarkers, Models Biological, Disease Progression) that the centrality analysis revealed as behaving generically. A few methodological terms still appear low in the rankings (Cells Cultured, Dose-Response Relationship). The set is explicit and can be extended, but stripping further trades defensibility for marginal cleanliness once nothing dominates.

**Co-occurrence is association, not causation.** Two descriptors appear together because they are studied together, not because they are biologically or causally linked. The network shows what is researched jointly, nothing more.

**Descriptor names are used as-is, with no disambiguation across MeSH revisions.** NLM occasionally renames or restructures descriptors over the 30-year span. A subject studied under an older descriptor name and a newer one would appear as two separate nodes. No attempt is made to reconcile these.

**The depth shift makes raw density incomparable across 2019/2020.** Mean topical descriptors per article falls from about 9 before 2020 to about 6 after (visible in the per-era headers), because of the NLM automated-indexing change documented in the EDA. Raw co-occurrence counts are therefore not comparable across that boundary. All cross-era reading is done on Jaccard, a ratio, which is far less sensitive to depth than raw counts. The era split itself is placed on this boundary for that reason.

**The early-era and recent-era samples differ in size and depth.** The three eras hold roughly 1.32M, 1.0M, and 0.74M articles respectively, with declining mean depth. The recent era has both fewer articles and fewer descriptors each, so its co-occurrence counts are thinner. This does not affect Jaccard much but does mean a pair needs proportionally more of the recent corpus to reach the same count.

💡 **Next Up:** Proceed to [`05_coauthorship_network.ipynb`](05_coauthorship_network.ipynb) for the co-authorship graph and collaboration structure. **Restrict to post-2014** (affiliation reliability) and note that author names are not disambiguated.